In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the dataset
data = pd.read_csv('crop_recommendation.csv')

# Features and target
X = data[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = data['label']

# Encode target variable (if needed)
y = pd.factorize(y)[0]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train SVM model
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train, y_train)

# Predictions
y_pred = svm_model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Print results
print("Support Vector Machine Model:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Support Vector Machine Model:
Accuracy: 0.9614
Precision: 0.9673
Recall: 0.9614
F1 Score: 0.9612


In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, ReLU, Dropout, Flatten, Dense, GlobalAveragePooling1D

# Load the dataset
data = pd.read_csv('crop_recommendation.csv')

# Features and target
X = data[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].values
y = data['label']

# Encode target variable (if needed)
y, labels = pd.factorize(y)

# Reshape the input for TCN (add a time dimension)
X = X[:, :, np.newaxis]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build the TCN model
def build_tcn_model(input_shape, num_classes):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(128, kernel_size=5, padding="causal", activation=None),
        BatchNormalization(),
        ReLU(),
        Dropout(0.4),
        Conv1D(128, kernel_size=5, padding="causal", activation=None),
        BatchNormalization(),
        ReLU(),
        Dropout(0.4),
        Conv1D(256, kernel_size=3, padding="causal", activation=None),
        BatchNormalization(),
        ReLU(),
        GlobalAveragePooling1D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

# Initialize the model
input_shape = (X_train.shape[1], X_train.shape[2])
num_classes = len(labels)
tcn_model = build_tcn_model(input_shape, num_classes)

# Compile the model
tcn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Train the model
tcn_model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=1)

# Save the trained TCN model
tcn_model.save('tcn_model.h5')

# Evaluate the model on test data
y_pred_probs = tcn_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Print results
print("Temporal Convolutional Network (TCN) Model:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Epoch 1/100
88/88 [==============================] - 4s 16ms/step - loss: 2.5017 - accuracy: 0.2095 - val_loss: 3.2556 - val_accuracy: 0.0994
Epoch 2/100
88/88 [==============================] - 1s 13ms/step - loss: 1.6846 - accuracy: 0.4247 - val_loss: 1.1480 - val_accuracy: 0.6193
Epoch 3/100
88/88 [==============================] - 1s 14ms/step - loss: 1.1861 - accuracy: 0.5582 - val_loss: 0.5617 - val_accuracy: 0.7812
Epoch 4/100
88/88 [==============================] - 1s 14ms/step - loss: 0.9097 - accuracy: 0.6477 - val_loss: 0.4474 - val_accuracy: 0.8750
Epoch 5/100
88/88 [==============================] - 1s 15ms/step - loss: 0.7134 - accuracy: 0.7287 - val_loss: 0.4232 - val_accuracy: 0.8125
Epoch 6/100
88/88 [==============================] - 2s 23ms/step - loss: 0.6476 - accuracy: 0.7337 - val_loss: 0.2996 - val_accuracy: 0.8665
Epoch 7/100
88/88 [==============================] - 1s 13ms/step - loss: 0.6022 - accuracy: 0.7536 - val_loss: 0.2992 - val_accuracy: 0.8778
Epoch 

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

# Load the trained model (replace 'tcn_model.h5' with your model file path)
model = tf.keras.models.load_model('tcn_model.h5')

# Labels (same order as used during training)
labels = ['rice', 'maize', 'chickpea', 'kidneybeans', 'pigeonpeas', 'mothbeans', 'mungbean', 'blackgram',
          'lentil', 'pomegranate', 'banana', 'mango', 'grapes', 'watermelon', 'muskmelon', 'apple',
          'orange', 'papaya', 'coconut', 'cotton', 'jute', 'coffee']

# Function to predict the crop
def predict_crop(n, p, k, temperature, humidity, ph, rainfall):
    # Create an input array
    input_data = np.array([[n, p, k, temperature, humidity, ph, rainfall]])
    input_data = input_data[:, :, np.newaxis]  # Add time dimension

    # Get prediction probabilities
    prediction_probs = model.predict(input_data)
    
    # Find the label with the highest probability
    predicted_label = labels[np.argmax(prediction_probs)]
    confidence = np.max(prediction_probs) * 100

    return predicted_label, confidence

# Example input (replace these with user inputs or dynamic values)
nitrogen = 12
phosphorus = 76
potassium = 80
temperature = 90.0  # in Celsius
humidity = 20.0  # in %
ph = 9.5
rainfall = 100.0  # in mm

# Get prediction
crop, confidence = predict_crop(nitrogen, phosphorus, potassium, temperature, humidity, ph, rainfall)

# Print the result
print(f"Recommended Crop: {crop}")
print(f"Confidence Level: {confidence:.2f}%")


1/1 [==============================] - 0s 312ms/step
Recommended Crop: papaya
Confidence Level: 81.44%
